# NB_hospital_daily_summary

Builds gold table gold.hospital_daily_summary` (one row per hospital per day with lab activity) from `silver.lab_results`, enriched with `silver.hospitals` and `silver.calendar`, including a day-over-day LAG trend, per the Silver-to-Gold STTM. Load type: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "adb_ws_wecare", "Catalog")
catalog = dbutils.widgets.get("catalog")
silver, gold = f"{catalog}.silver", f"{catalog}.gold"
target = f"{gold}.hospital_daily_summary"
print(f"Target: {target}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

spark.sql(f"""
CREATE OR REPLACE TABLE {target}
COMMENT 'Gold: one row per hospital per day of lab activity with day-over-day trend (STTM Silver to Gold)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                               AS summary_date,
    COUNT(*)                                AS total_tests,
    COUNT(DISTINCT patient_id)              AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)               AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(t.hospital_id AS INT)                                      AS hospital_id,
  CAST(h.hospital_name AS STRING)                                 AS hospital_name,
  CAST(t.summary_date AS DATE)                                    AS summary_date,
  CAST(c.day_name AS STRING)                                      AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                   AS is_weekend,
  CAST(t.total_tests AS INT)                                      AS total_tests,
  CAST(t.unique_patients_tested AS INT)                           AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                              AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                              AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                              AS critical_test_count,
  CAST(t.abnormal_test_count / t.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(t.critical_test_count / t.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                             AS prev_day_total_tests,
  CAST(CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
            ELSE (t.total_tests - t.prev_day_total_tests) / t.prev_day_total_tests * 100
       END AS DECIMAL(6,2))                                       AS test_volume_change_pct
FROM trend t
LEFT JOIN {silver}.hospitals h ON t.hospital_id = h.hospital_id
LEFT JOIN {silver}.calendar  c ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)
""")
print(f"Rebuilt {target}")

In [0]:
updated_df = spark.table(target)
record_count = updated_df.count()
print(f"{target}: {record_count} records")
dbutils.notebook.exit(str(record_count))